In [21]:
!pip install ollama
!pip install openpyxl
import pandas as pd

def chat_with_ollama(model, message_content):
    return ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': message_content}]
    )

df = pd.read_excel("LLM_Experiment.xlsx")
df.columns = ["report", "label"]

df["label"] = df["label"].replace(["", " ", "nan", "None"], pd.NA)

print("原始資料筆數：", len(df))
print(df)

MODEL_NAME = "env-labeler"

labeled_df = df[df["label"].notna()].copy()
unlabeled_df = df[df["label"].isna()].copy()

print("已標註筆數：", len(labeled_df))
print("未標註筆數：", len(unlabeled_df))

target_df = unlabeled_df.tail(2).copy()

example_text = ""
for i, row in enumerate(labeled_df.head(3).itertuples(index=False), start=1):
    example_text += f"""範例{i}
報告：
{row.report}

標註：
{row.label}

"""

results = []

for idx, row in target_df.iterrows():
    txt = f"""
你是一個環境稽查報告標註助手。

你的工作是找出「違規或被處理的廢棄物」及其事件時間。

【輸出格式】
只能輸出：
(代碼)YYYY/M
(代碼)YYYY/M-YYYY/M
(null)

【規則】
1. 只標註「違規、被處理、被要求改善」的廢棄物
2. 不要標註背景資料或許可資訊
3. 不要標註再利用登記、核准期間
4. 不要輸出數量（公噸、kg）

5. 時間規則：
    民國年 + 1911 = 西元年
    99年 = 2010年
    102年 = 2013年
    103年 = 2014年
    104年 = 2015年
    轉為 YYYY/M

6. 若只有單一日期 → 只輸出 YYYY/M
7. 不可以自己亂補時間區間
8. 若只有單一時間點，不可以輸出「-」的符號

9. 若報告內容為：
    尚無明顯異常情事
    無作業跡象
    僅歇業或遷廠
   → 必須輸出 (null)

10. 若沒有明確違規事件 → 輸出 (null)

===== 範例 =====
{example_text}

===== 新資料 =====
報告：
{row['report']}

請只輸出一行標註結果：
"""

    output = chat_with_ollama(MODEL_NAME, txt)
    result = output['message']['content'].replace('<', '').strip()

    print("=" * 100)
    print(f"第 {idx+1} 筆未標註資料")
    print("=" * 100)
    print("原始報告：")
    print(row["report"])
    print("\n模型生成的標註：")
    print(result)
    print("\n")

    results.append({
        "report": row["report"],
        "generated_label": result
    })

result_df = pd.DataFrame(results)
result_df

原始資料筆數： 5
                                              report                  label
0  一、環境督察大隊及內政部警政署保安警察第七總隊第三大隊第三中隊至該公司督察有關臺南市遭棄置廢...  (R-0201)2017/7-2018/7
1  "1.巡查類別：例行稽查。2.事業廢棄物網路申報：符合規定。3.清除事業廢棄物：符合規定。4...                 (null)
2  1.本日至該公司查核事業廢棄物再利用者登記檢核申請查核。\n2.該公司從事漿紙原料及塑膠粒原...                 (null)
3  一、依據行政院環境保護署督字第H號函辦理稽查。\n二、經查該廠係從事廢塑膠(R-0201)再...                    NaN
4  一、現場依規定出示檢查證件，表明身分並說明來意。\n二、本日係執行環保署EPx稽查專案。\n...                    NaN
已標註筆數： 3
未標註筆數： 2
第 4 筆未標註資料
原始報告：
一、依據行政院環境保護署督字第H號函辦理稽查。
二、經查該廠係從事廢塑膠(R-0201)再利用機構(公告每月98公噸)及經濟部個案再利用廢塑膠混合物(D-0299每月20噸)再利用事業名稱N公司。
三、關於M公司違反廢清法第38條第一項規定，退關出倉後一批(電纜披露之交連PE廢棄物)，於本縣，惟查該廠並無廢交聯PE(D-0299)再利用許可相關證明文件，另查退關該批廢棄物(D-0299)於99年11月11日車號XXX，貨櫃型式40呎，據業者指稱總重為38410kg，並放置於廠內及廠區，已於現場拍照存證。
四、仍當場責成業者，於未取得D-0299通案再利用許可，不得收受再利用；據業者指稱本公司有再利用身份檢後(公告廢塑膠經濟部工業局事業廢棄物再利用種類及管理方式)編號十、廢塑膠(最大再利用量98公噸)與D-0299同性質，為釐清案情及另向行政院環保署北區督察大隊查詢並視情節依法辦理，也當場責請令業者該批廢棄物(D-0299)暫存不動。
五、上開紀錄交由業者閱覽後，確認無誤後，始簽名於後。

模型生成的標註：
(D-0299)2010/11


第 5 筆未標註資料
原始報告：
一、現場

,report,generated_label
0,一、依據行政院環境保護署督字第H號函辦理稽查。\n二、經查該廠係從事廢塑膠(R-0201)再...,(D-0299)2010/11
1,一、現場依規定出示檢查證件，表明身分並說明來意。\n二、本日係執行環保署EPx稽查專案。\n...,(null)
